# 05 · From Sequence Models to Attention — building a TinyGPT

Notebook 04 fed a **bag of characters** into an MLP: reorder the 5 input
characters and the prediction cannot tell the difference. Real language models
need the opposite: every token should **look back** at the tokens before it and
decide *which ones matter*. That mechanism is **self-attention**, and stacking
it gives you a Transformer — the architecture behind GPT.

This notebook builds one from scratch in three steps:

1. **Single-head self-attention** from first principles (NumPy first, then PyTorch) — §2
2. **Stacked Transformer blocks**: multi-head attention + FFN + residuals + LayerNorm — §3
3. **Training** the resulting tiny GPT on short sequences (64 tokens) of Tiny Shakespeare — §4

By the end we generate Shakespeare-like text the same way notebook 04 did, but
with a 64-token context instead of a 5-character one.


## 1. Data & batches (same corpus as notebook 04)

Same character-level vocabulary as before (65 characters). The one real change
is **how batches are built**: instead of one 5-char window → one next
character, we now train on the whole chunk at once. For a chunk
`x = tokens[i:i+T]` the target is simply `y = tokens[i+1:i+1+T]`, so a single
sequence of T=64 tokens yields **64 parallel next-token predictions**.


In [ ]:
import math
import os
import urllib.request

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


In [ ]:
def load_shakespeare():
    """Reuse the local copy if it exists, otherwise download Tiny Shakespeare."""
    for path in ("../dataset/input.txt", "dataset/input.txt", "input.txt"):
        if os.path.exists(path):
            print(f"Loaded dataset from: {path}")
            with open(path, "r", encoding="utf-8") as f:
                return f.read()
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    print(f"Local copy not found, downloading: {url}")
    with urllib.request.urlopen(url) as r:
        return r.read().decode("utf-8")


text = load_shakespeare()
print(f"Dataset length: {len(text):,} characters")
print(text[:200] + "...")


In [ ]:
chars = sorted(set(text))
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}
vocab_size = len(chars)

print(f"Character vocab size: {vocab_size}")

# Same encoding as notebook 04, but as one long flat sequence:
# a language model only needs (tokens[0..n-2]) -> (tokens[1..n-1]).
data = torch.tensor([char_to_idx[c] for c in text], dtype=torch.long)

# 90/10 train/validation split
n_train = int(0.9 * len(data))
train_data = data[:n_train]
val_data = data[n_train:]
print(f"Train tokens: {len(train_data):,} | Val tokens: {len(val_data):,}")


In [ ]:
def get_batch(source, batch_size=32, block_size=64):
    """Sample random chunks: predict every next token in parallel.

    x: (B, T) the current tokens
    y: (B, T) the same tokens shifted by one -> B*T predictions per batch
    (notebook 04 predicted ONE next token per 5-char context.)
    """
    ix = torch.randint(len(source) - block_size - 1, (batch_size,))
    x = torch.stack([source[i : i + block_size] for i in ix])
    y = torch.stack([source[i + 1 : i + 1 + block_size] for i in ix])
    return x.to(device), y.to(device)


xb, yb = get_batch(train_data)
print(f"x: {tuple(xb.shape)}  y: {tuple(yb.shape)}")
print("context:", repr("".join(idx_to_char[i.item()] for i in xb[0, :24])))
print("target :", repr("".join(idx_to_char[i.item()] for i in yb[0, :24])))


## 2. Task 1 — single-head self-attention from scratch

**The problem with notebook 04's MLP:** `fc1` sees a flattened one-hot vector.
Shuffle the 5 characters and the same weights hit the same inputs — the model
is *permutation-symmetric* and has no notion of "which character should I look
at". For language we want the opposite: position-aware, selective mixing.

**Self-attention** gives each token a *query* (what am I looking for?), a *key*
(what do I contain?) and a *value* (what do I contribute if selected?). Token
`t` compares its query against every key `t' <= t` and mixes the values
proportionally:

    Attention(Q, K, V) = softmax( Q Kᵀ / √d_k ) V

The mask (upper triangle set to −∞ *before* the softmax) makes it **causal**:
token `t` can only attend to positions <= t — exactly what next-token
prediction needs. The `√d_k` keeps dot products from growing with dimension,
which would saturate the softmax.

We start in NumPy so every matrix is visible, then port it to PyTorch.


In [ ]:
def softmax_np(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)   # numerical stability
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)


def single_head_attention_np(x, Wq, Wk, Wv, causal=True):
    """
    x : (T, d)          input embeddings, one row per token
    Wq, Wk, Wv : (d, d_k)  learned projections
    returns (out, attn):
        out  (T, d_k) : each output row mixes information from rows <= t
        attn (T, T)   : attention weights (rows sum to 1)
    """
    Q, K, V = x @ Wq, x @ Wk, x @ Wv             # (T, d_k) each
    scores = Q @ K.T / math.sqrt(K.shape[-1])    # (T, T) raw affinities
    if causal:
        T = x.shape[0]
        future = np.triu(np.ones((T, T), dtype=bool), k=1)
        scores = np.where(future, -np.inf, scores)  # forbid looking ahead
    attn = softmax_np(scores, axis=-1)
    return attn @ V, attn


In [ ]:
T, d, d_k = 6, 8, 8
rng = np.random.default_rng(0)
x_demo = rng.normal(size=(T, d))
Wq = rng.normal(size=(d, d_k)) / math.sqrt(d)
Wk = rng.normal(size=(d, d_k)) / math.sqrt(d)
Wv = rng.normal(size=(d, d_k)) / math.sqrt(d)

out_demo, A_demo = single_head_attention_np(x_demo, Wq, Wk, Wv)

print("Attention matrix A (rows = query token, cols = key token):")
print(np.round(A_demo, 2))
print()
print("Every row sums to 1          :", np.allclose(A_demo.sum(axis=-1), 1.0))
print("Upper triangle is exactly 0  :", A_demo[np.triu_indices(T, k=1)].sum() == 0.0)
print("Output shape                 :", out_demo.shape)


Read the matrix like this: **row `t` = where token `t` looks** (columns = key
tokens). With random weights the attention is roughly uniform — nothing
interesting yet. Training is what makes rows become sharp and structured (e.g.
strong weight onto the token that starts the current word). The upper triangle
is exactly 0: with a causal mask, the future is invisible.

Now the same thing in PyTorch, as an `nn.Module` (batched, learnable,
autograd-ready):


In [ ]:
class SingleHeadSelfAttention(nn.Module):
    """Causal single-head self-attention, written from first principles."""

    def __init__(self, d_model, d_head=None):
        super().__init__()
        d_head = d_head or d_model
        self.q_proj = nn.Linear(d_model, d_head, bias=False)
        self.k_proj = nn.Linear(d_model, d_head, bias=False)
        self.v_proj = nn.Linear(d_model, d_head, bias=False)
        # True where attention is FORBIDDEN (strict upper triangle)
        self.register_buffer(
            "mask", torch.triu(torch.ones(512, 512, dtype=torch.bool), diagonal=1)
        )

    def forward(self, x):
        B, T, _ = x.shape
        Q, K, V = self.q_proj(x), self.k_proj(x), self.v_proj(x)   # (B, T, d_head)

        scores = Q @ K.transpose(-2, -1) / math.sqrt(K.size(-1))   # (B, T, T)
        scores = scores.masked_fill(self.mask[:T, :T], float("-inf"))
        attn = F.softmax(scores, dim=-1)                           # rows sum to 1

        return attn @ V, attn


In [ ]:
torch.manual_seed(0)
attn_layer = SingleHeadSelfAttention(d_model=64)

tokens = xb[:4]                                   # (B=4, T=64) character indices
emb_demo = nn.Embedding(vocab_size, 64)
x_in = emb_demo(tokens)                           # (B, T, 64) learned embeddings

out, weights = attn_layer(x_in)
print(f"in : {tuple(x_in.shape)} -> out: {tuple(out.shape)}")
print(f"attention weights: {tuple(weights.shape)}")

row_sums = weights.sum(dim=-1)
print("rows sum to 1:", torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-5))
print("token 0 sees the future? attention mass =", weights[:, 0, 1:].sum().item(), "(0 = causal)")


In [ ]:
plt.figure(figsize=(5, 5))
plt.imshow(weights[0].detach().cpu().numpy(), cmap="viridis")
plt.title("Single-head attention (random weights, batch item 0)")
plt.xlabel("Key position (token being looked at)")
plt.ylabel("Query position (token doing the looking)")
plt.colorbar(label="attention weight")
plt.show()


## 3. Task 2 — from one head to stacked Transformer blocks

One attention head gives one "relevance pattern". Real transformers add:

- **multiple heads** — each learns its own pattern (some track spaces/newlines,
  some the current speaker, ...);
- a small **feed-forward network** per position — the "thinking" step after the
  "talking" step;
- **residual connections** around both — gradients flow and deep stacks train;
- **LayerNorm** for stable activation scales;
- **positional embeddings** — attention itself is order-blind.

### 3.1 Multi-head attention

Instead of `h` small projections, run one big projection and split it into
`h` heads of size `d_model / h`:


In [ ]:
class MultiHeadSelfAttention(nn.Module):
    """h single heads in parallel via one big projection each."""

    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model)   # lets heads interact

        self.attn_drop = nn.Dropout(dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(512, 512, dtype=torch.bool), diagonal=1)
        )

    def forward(self, x):
        B, T, C = x.shape

        # (B, T, C) -> (B, n_heads, T, d_head)
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        # scaled dot-product attention per head
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_head)   # (B, h, T, T)
        scores = scores.masked_fill(self.mask[:T, :T], float("-inf"))
        attn = self.attn_drop(F.softmax(scores, dim=-1))

        y = attn @ v                                     # (B, h, T, d_head)
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-merge the heads
        return self.out_proj(y)


### 3.2 The block: attention + feed-forward, both in residuals

A Transformer block has two sub-steps. Attention lets tokens **talk to each
other**; the FFN then lets each token **process** what it collected,
independently per position. Residuals (`x + f(x)`) mean each sub-step only
needs to learn a small *correction* — that is what makes deep stacks trainable.
Pre-LayerNorm (normalize *before* each sub-step) keeps scales steady.


In [ ]:
class TransformerBlock(nn.Module):
    """Communication (attention) + computation (FFN), both wrapped in residuals."""

    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),   # wider inner layer (standard 4x)
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # residual 1
        x = x + self.ffn(self.ln2(x))    # residual 2
        return x


block = TransformerBlock(d_model=64, n_heads=4)
x_test = torch.randn(2, 32, 64)
print(f"block: {tuple(x_test.shape)} -> {tuple(block(x_test).shape)}")
print("parameters in one block:", sum(p.numel() for p in block.parameters()))


### 3.3 The full model: embedding + stacked blocks + LM head

Assembly order for a decoder-only GPT:

    token ids
      -> token embedding + positional embedding
      -> N x TransformerBlock   (each causal, so the whole stack stays causal)
      -> final LayerNorm
      -> Linear projection to vocab logits

The table that maps hidden states to vocabulary logits is **tied** to the
token-embedding matrix (same weights used in both directions) — a standard
trick that saves parameters and works well at this scale.


In [ ]:
class TinyGPT(nn.Module):
    """A minimal decoder-only transformer (GPT-style) built from the blocks above."""

    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=4,
                 block_size=64, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            [TransformerBlock(d_model, n_heads, dropout) for _ in range(n_layers)]
        )
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

        self.lm_head.weight = self.tok_emb.weight   # weight tying

        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)

        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))   # (B, T, C)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)

        logits = self.lm_head(x)                               # (B, T, vocab)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)), targets.reshape(-1)
            )
        return logits, loss


In [ ]:
model = TinyGPT(vocab_size, d_model=128, n_heads=4, n_layers=4, block_size=64).to(device)

n_params = sum(p.numel() for p in model.parameters())
n_unique = n_params - model.tok_emb.weight.numel()   # the tied table is counted twice
print(f"Parameters (tied table counted twice): {n_params:,}")
print(f"Parameters (unique):                   {n_unique:,}")

logits, loss = model(*get_batch(train_data))
print(f"logits: {tuple(logits.shape)}   initial loss: {loss.item():.4f}")
print(f"uniform-guessing loss would be: {math.log(vocab_size):.4f}")


## 4. Task 3 — training on short sequences (32–64 tokens)

Standard recipe: AdamW, gradient clipping, periodic evaluation on the held-out
slice. Note we **never shuffle into epochs** like notebook 04 — each step
samples a fresh random chunk of 64 tokens, which is simpler and works well for
language models.

Expected behaviour: the loss starts near `ln(65) ≈ 4.17` (uniform guessing)
and drops to roughly **1.5–1.8** within a few thousand steps. `BLOCK_SIZE = 64`
here; set it to 32 for a faster first run (see §6 for what to compare).


In [ ]:
# --- Training configuration --------------------------------------------------
BLOCK_SIZE = 64      # context length in tokens (try 32 for faster runs)
BATCH_SIZE = 64
LEARNING_RATE = 3e-4
MAX_STEPS = 3000     # ~5-10 min on CPU; use 500 for a quick smoke run
EVAL_INTERVAL = 300
EVAL_ITERS = 50      # batches averaged for a more stable loss estimate

model = TinyGPT(vocab_size, d_model=128, n_heads=4, n_layers=4,
                block_size=BLOCK_SIZE, dropout=0.1).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
print(f"Training on sequences of {BLOCK_SIZE} tokens, batch of {BATCH_SIZE}")


In [ ]:
@torch.no_grad()
def estimate_loss(model, iters=EVAL_ITERS):
    """Average loss over fresh batches - a less noisy read on progress."""
    model.eval()
    out = {}
    for name, source in (("train", train_data), ("val", val_data)):
        losses = torch.zeros(iters)
        for k in range(iters):
            xb, yb = get_batch(source, BATCH_SIZE, BLOCK_SIZE)
            _, loss = model(xb, yb)
            losses[k] = loss.item()
        out[name] = losses.mean().item()
    model.train()
    return out


train_losses, val_losses = [], []
model.train()

for step in range(1, MAX_STEPS + 1):
    xb, yb = get_batch(train_data, BATCH_SIZE, BLOCK_SIZE)

    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # keep updates sane
    optimizer.step()

    train_losses.append(loss.item())

    if step % EVAL_INTERVAL == 0 or step == MAX_STEPS:
        losses = estimate_loss(model)
        val_losses.append(losses["val"])
        print(f"step {step:5d} | train {losses['train']:.4f} | val {losses['val']:.4f}")


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_losses, color="tab:blue", alpha=0.3, label="train (per step)")

k = 50   # smoothing window
smooth = np.convolve(train_losses, np.ones(k) / k, mode="valid")
plt.plot(range(k - 1, len(train_losses)), smooth, color="tab:blue", label="train (smoothed)")

eval_steps = list(range(EVAL_INTERVAL, len(train_losses) + 1, EVAL_INTERVAL))
plt.plot(eval_steps[:len(val_losses)], val_losses, "o-", color="tab:orange", label="val")

plt.xlabel("training step")
plt.ylabel("cross-entropy loss")
plt.title(f"TinyGPT loss (block_size={BLOCK_SIZE}, {MAX_STEPS} steps)")
plt.legend()
plt.show()


## 5. Generating text

Same autoregressive loop as notebook 04: predict → append → repeat. Two
differences: the context window is now up to `block_size` tokens (not 5
characters), and `top_k` sampling is available to cut off the long tail of
unlikely characters.


In [ ]:
@torch.no_grad()
def generate(model, start_str="First Citizen:", max_new_tokens=400,
             temperature=0.8, top_k=None):
    """Sample from the model, one character at a time."""
    model.eval()
    idx = torch.tensor([[char_to_idx[c] for c in start_str]],
                       dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.block_size:]        # crop to the context window
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature      # only the LAST position matters

        if top_k is not None:                        # keep only the top-k logits
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float("-inf")

        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)

    return "".join(idx_to_char[i.item()] for i in idx[0])


print(generate(model, "First Citizen:", max_new_tokens=400, temperature=0.8))


In [ ]:
for prompt in ("JULIET:", "KING RICHARD III:", "\n"):
    print("=" * 60)
    print(repr(prompt))
    print("-" * 60)
    print(generate(model, prompt, max_new_tokens=250, temperature=0.8))


In [ ]:
print("temperature = 0.3 (focused, repetitive):")
print(generate(model, "First Citizen:", max_new_tokens=250, temperature=0.3))
print()
print("temperature = 1.2 (creative, chaotic):")
print(generate(model, "First Citizen:", max_new_tokens=250, temperature=1.2))


## 6. Exercises

1. **Context length**: set `BLOCK_SIZE = 32`, retrain, compare val loss and
   text quality with 64. Where exactly does the longer context help?
2. **Depth**: `n_layers = 1` vs 4 — how does the val-loss curve change?
3. **Heads**: `n_heads = 1` vs 4 at fixed `d_model` — does multi-head win here?
4. **Look inside a trained head** (compare with the random-weight plot in §2):

   ```python
   with torch.no_grad():
       idx = xb[:1]                                   # one batch row
       x = model.tok_emb(idx) + model.pos_emb(torch.arange(idx.shape[1], device=device))
       _, w = model.blocks[0].attn(model.blocks[0].ln1(x))
       plt.imshow(w[0, 0].cpu())                      # layer 1, head 0
   ```
5. **Regularization**: dropout 0.0 vs 0.2 — watch the train/val gap.
6. **Sampling**: try `top_k=10` vs `None` in §5 — which text reads better?


## 7. Key takeaways

- Self-attention is a **dynamic, content-based weighted average** over previous
  tokens; an MLP's fixed weights cannot do this.
- The **causal mask** is what turns a Transformer into a language model:
  predict every next token in parallel, but never see the future.
- A Transformer **block** = communication (attention) + computation (FFN),
  both wrapped in residuals and LayerNorm; stack N of them → GPT.
- Batches of short sequences give **T parallel predictions** per sequence —
  the shift from notebook 04's "5 chars → 1 next char".
- Natural next steps: swap characters for BPE subwords (see notebook 04),
  encoder-decoder cross-attention, sinusoidal/RoPE position encodings,
  learning-rate schedules, and scaling up.
